# Expectation & Variance — companion notebook

Runnable companion for [`expectation-variance.md`](./expectation-variance.md).

1. Demonstrate that **linearity of expectation holds even for dependent variables**.
2. Monte Carlo check of $\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y) + 2\,\text{Cov}(X, Y)$.
3. Visualize how the sample variance estimator converges as $n$ grows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Linearity holds without independence

Construct a strongly **dependent** pair: $X \sim \mathcal{N}(0, 1)$ and $Y = X^3 - X$. Independence clearly fails, yet $\mathbb{E}[X + Y] = \mathbb{E}[X] + \mathbb{E}[Y]$ holds exactly in expectation.

In [ ]:
n = 200_000
X = rng.standard_normal(n)
Y = X**3 - X   # deterministic function of X => maximally dependent

EX, EY, EXY = X.mean(), Y.mean(), (X + Y).mean()
print(f'E[X]       = {EX:+.4f}   (true 0)')
print(f'E[Y]       = {EY:+.4f}   (true E[X^3]-E[X] = 0)')
print(f'E[X+Y]     = {EXY:+.4f}')
print(f'E[X]+E[Y]  = {EX+EY:+.4f}   (matches E[X+Y] up to MC noise)')

## 2. $\text{Var}(X+Y)$ — the covariance cross-term matters

Use a bivariate Normal with correlation $\rho$. Analytically $\text{Var}(X+Y) = \sigma_X^2 + \sigma_Y^2 + 2\rho\sigma_X\sigma_Y$.

In [ ]:
sx, sy = 1.0, 2.0
print(f'{"rho":>5} | {"analytic":>10} | {"MC est":>10}')
print('-' * 35)
for rho in [-0.8, -0.3, 0.0, 0.5, 0.95]:
    cov = np.array([[sx**2, rho*sx*sy], [rho*sx*sy, sy**2]])
    samples = rng.multivariate_normal([0, 0], cov, size=200_000)
    Xs, Ys = samples[:, 0], samples[:, 1]
    analytic = sx**2 + sy**2 + 2*rho*sx*sy
    mc = np.var(Xs + Ys, ddof=1)
    print(f'{rho:>5.2f} | {analytic:>10.4f} | {mc:>10.4f}')

## 3. Plot — sample variance estimator convergence

For $X \sim \mathcal{N}(0, 1)$, the sample variance $\hat\sigma^2_n$ should converge to 1 as $n$ grows. Compare $\frac{1}{n}\sum$ (biased) vs $\frac{1}{n-1}\sum$ (unbiased).

In [ ]:
ns = np.unique(np.logspace(0.7, 4, 40).astype(int))   # 5 .. 10000
n_trials = 400
biased = np.zeros((len(ns), n_trials))
unbiased = np.zeros((len(ns), n_trials))
for i, n in enumerate(ns):
    samples = rng.standard_normal((n_trials, n))
    biased[i] = samples.var(axis=1, ddof=0)
    unbiased[i] = samples.var(axis=1, ddof=1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(1.0, color='black', ls='--', lw=1, label='true variance = 1')
ax.semilogx(ns, biased.mean(axis=1), marker='o', ms=3, label='E[ddof=0]  (biased, ~1 - 1/n)')
ax.semilogx(ns, unbiased.mean(axis=1), marker='s', ms=3, label='E[ddof=1]  (unbiased)')
ax.set_xlabel('sample size n'); ax.set_ylabel('mean estimator value across trials')
ax.set_title('Bessel correction: ddof=1 is unbiased, ddof=0 underestimates')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()